# Quickstart: SHARP Extrapolation By Date

This notebook downloads one SHARP magnetogram and runs a simple cartesian extrapolation.

Workflow:
1. Edit the parameter cell below.
2. Run all cells.
3. Use the exported files from the output directory.

In [ ]:
# Edit only this cell
JSOC_EMAIL = "your_email@example.com"
DATE = "2024-05-07T17:24:00"
NOAA_AR = 13664
OUTPUT_NAME = "sharp_quickstart_13664"
EPOCHS = 20
EXPORT_FORMATS = ["vtk", "npz"]

In [ ]:
from copy import deepcopy
from datetime import datetime
from pathlib import Path

from nf2.core.runner import run
from nf2.data.download import download_SHARP_series
from nf2.export.core import export_checkpoint
from nf2.train.util import load_yaml_config

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

download_dir = repo_root / "data" / "notebooks" / OUTPUT_NAME / "sharp"
download_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.fromisoformat(DATE)
download_SHARP_series(download_dir=str(download_dir), email=JSOC_EMAIL, t_start=timestamp, noaa_num=NOAA_AR)

config = load_yaml_config(str(repo_root / "config" / "sharp" / "377.yaml"))
config["base_path"] = str(repo_root / "results" / "notebooks" / OUTPUT_NAME)
config["work_directory"] = str(repo_root / "work" / "notebooks" / OUTPUT_NAME)
config["logging"] = deepcopy(config.get("logging", {}))
config["logging"]["offline"] = True
config["training"] = deepcopy(config.get("training", {}))
config["training"]["epochs"] = EPOCHS
config["data"] = deepcopy(config.get("data", {}))
config["data"]["train_configs"] = [{
    "type": "sharp",
    "ds_id": "boundary_01",
    "data_path": str(download_dir),
}]
config["data"]["valid_configs"] = [
    {"type": "sharp", "ds_id": "sharp_boundary", "data_path": str(download_dir), "plot": False},
    {"type": "cube", "ds_id": "cube"},
    {"type": "slices", "ds_id": "slices"},
]

config

In [ ]:
run(**config)

In [ ]:
checkpoint_path = Path(config["base_path"]) / "extrapolation_result.nf2"
exports = {fmt: export_checkpoint(str(checkpoint_path), fmt) for fmt in EXPORT_FORMATS}
checkpoint_path, exports